In [1]:
# Imports
from matplotlib.ticker import FixedLocator, FuncFormatter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
# Load data
df = pd.read_csv('./data/FinalTransactionReport.csv')

In [3]:
from matplotlib.ticker import FixedLocator, FuncFormatter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('./data/FinalTransactionReport.csv')

# Ensure timestamps are datetime
df['Event Timestamp'] = pd.to_datetime(df['Event Timestamp'])

# ---- Game-level aggregation ----
game_level = (
    df.groupby('Event ID')
    .apply(lambda x: pd.Series({
        # Event metadata
        "event_id": x['Event ID'].iloc[0],
        "event_name": x['Event Name'].iloc[0],
        "month": x['Event Timestamp'].iloc[0].strftime("%B"),
        "day_of_week": x['Day of Week'].iloc[0],
        "is_weekend": int(x['Day of Week'].iloc[0] in ["Friday","Saturday","Sunday"]),

        # Demand outcomes
        "total_tickets_sold": x['Tickets Sold'].sum(),
        "total_revenue": x['Sales Total'].sum(),

        # Demand timing
        "avg_days_before_purchase": x['Days Before Event'].mean(),

        # Weather
        "temperature_f": x['temperature_f'].iloc[0],
        "humidity": x['humidity'].iloc[0],
        "weather_category": x['weather_category'].iloc[0],
    }))
    .reset_index(drop=True)
)

# ---- Filter months ----
allowed_months = ["April","May","June","July","August","September"]
game_level = game_level[game_level["month"].isin(allowed_months)].copy()

# Keep month as ordered categorical (for interpretability)
game_level["month"] = pd.Categorical(
    game_level["month"],
    categories=allowed_months,
    ordered=True
)

# ---- Define relative demand target ----
avg_tickets = game_level["total_tickets_sold"].mean()
game_level["demand_pct_above_avg"] = (
    (game_level["total_tickets_sold"] - avg_tickets) / avg_tickets
)

# (Optional) Also keep a log target if you want to compare later
game_level["log_tickets"] = np.log1p(game_level["total_tickets_sold"])

game_level.head()


/tmp/ipykernel_3335/3982238753.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,event_id,event_name,month,day_of_week,is_weekend,total_tickets_sold,total_revenue,avg_days_before_purchase,temperature_f,humidity,weather_category,demand_pct_above_avg,log_tickets
0,15084,Salem Red Sox,April,Tuesday,0,1113,22988.13,12.225664,47.1,31,Clear,0.523614,7.015712
1,15085,Salem Red Sox,April,Wednesday,0,245,5287.52,5.203540,52.0,38,Clear,-0.664613,5.505332
2,15086,Salem Red Sox,April,Thursday,0,162,3760.72,9.597015,59.4,50,Cloudy,-0.778234,5.093750
3,15087,Salem Red Sox,April,Friday,1,231,5259.57,23.172840,46.2,97,Rainy,-0.683778,5.446737
4,15088,Salem Red Sox,April,Saturday,1,590,11894.36,7.490654,52.2,63,Cloudy,-0.192334,6.381816


In [4]:
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.model_selection import train_test_split

# ---- Feature set ----
# Let EBM handle categoricals natively (no one-hot)
feature_cols = [
    "event_name",
    "month",
    "is_weekend",
    "avg_days_before_purchase",
    "temperature_f",
    "humidity",
    "weather_category",
]

X = game_level[feature_cols]
y = game_level["demand_pct_above_avg"]  # relative demand strength

# ---- Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ---- Fit EBM ----
ebm = ExplainableBoostingRegressor(
    interactions=10,
    max_bins=256,
    outer_bags=1,      # <- disable bagging across processes
    inner_bags=0,
    learning_rate=0.01,
    max_rounds=5000,
    n_jobs=1,          # <- force single-process, avoids loky/joblib issues
    random_state=42
)

ebm.fit(X_train, y_train)

print("Train R^2:", ebm.score(X_train, y_train))
print("Test  R^2:", ebm.score(X_test, y_test))


Train R^2: 0.4962330958019765
Test  R^2: 0.39133631131997126


In [5]:
from interpret import show
from interpret.glassbox import ExplainableBoostingRegressor

global_expl = ebm.explain_global(name="Game-Level Demand Model")
show(global_expl)


<!-- http://127.0.0.1:7001/134596154290048/ -->